# Ejercicio · Algoritmos genéticos: mejora de parámetros (5 puntos)

**Objetivo:** proponer y defender una configuración de algoritmo genético que obtenga buenas soluciones con un esfuerzo razonable.

**No debes programar:** únicamente escribe tu número de lista y **modifica valores** de los parámetros marcados. El algoritmo, las tablas y las gráficas ya están implementados.

### Instrucciones
1. Ejecuta el cuaderno guiado antes de iniciar.
2. Introduce tu número de lista y conserva su valor en todas las corridas.
3. Usa como punto de partida la configuración BASE, que no debes alterar.
4. Diseña **tres configuraciones diferentes** (P1, P2, P3) cambiando por lo menos **dos parámetros** respecto a BASE en cada una.
5. Ejecuta las comparaciones, observa los resultados y escribe un reporte breve con evidencia numérica. **No se califica por obtener error cero**, sino por experimentar e interpretar.


In [ ]:
# ÚNICA EDICIÓN OBLIGATORIA ANTES DE CONFIGURAR TUS PROPUESTAS
NUMERO_LISTA = 12  # Cambia 12 por tu número de lista (1 a 100)


## 1. Motor del algoritmo genético (no modificar)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# FUNCIÓN OBJETIVO: minimizar f(x) = (x - a)^2
# El número de lista cambia a, pero conserva la dificultad del problema.
if not isinstance(NUMERO_LISTA, int) or not (1 <= NUMERO_LISTA <= 100):
    raise ValueError("NUMERO_LISTA debe ser un entero del 1 al 100")
a = 4.0 + 0.13 * NUMERO_LISTA
LIM_INF, LIM_SUP = -4.0, 21.0

def objetivo(x):
    return (np.asarray(x) - a) ** 2

# Rueda probabilística: como el problema es de MINIMIZACIÓN,
# transformamos el costo en una aptitud positiva: 1 / (1 + f(x)).
def elegir_padre(poblacion, costos, metodo, rng):
    if metodo == "torneo":
        i, j = rng.choice(len(poblacion), size=2, replace=False)
        return poblacion[i] if costos[i] <= costos[j] else poblacion[j]
    if metodo == "ruleta":
        pesos = 1.0 / (1.0 + costos)
        probabilidades = pesos / pesos.sum()
        return poblacion[rng.choice(len(poblacion), p=probabilidades)]
    raise ValueError("Selecciona 'torneo' o 'ruleta'")

def algoritmo_genetico(config, semilla, guardar_poblaciones=False):
    n = int(config["poblacion"])
    generaciones = int(config["generaciones"])
    pm = float(config["probabilidad_mutacion"])
    sigma = float(config["desviacion_mutacion"])
    elite = int(config["elitismo"])
    metodo = config["seleccion"]
    if not (4 <= n <= 100 and 1 <= generaciones <= 300 and 0 <= pm <= 1 and
            0 <= sigma <= 5 and 0 <= elite < n and metodo in ["torneo", "ruleta"]):
        raise ValueError("Configuración fuera de rango; revisa la tabla de parámetros")
    rng = np.random.default_rng(semilla)
    poblacion = rng.uniform(LIM_INF, LIM_SUP, size=n)
    historial = []
    poblaciones = []
    mejor_global = float("inf")
    mejor_x = None
    # En cada generación se evalúa la población completa.
    # Se registra la generación inicial (0) y las generaciones nuevas.
    for g in range(generaciones + 1):
        costos = objetivo(poblacion)
        i_mejor = int(np.argmin(costos))
        if float(costos[i_mejor]) < mejor_global:
            mejor_global = float(costos[i_mejor])
            mejor_x = float(poblacion[i_mejor])
        historial.append({"generacion": g,
                          "mejor_generacion": float(costos[i_mejor]),
                          "mejor_acumulado": mejor_global,
                          "promedio": float(np.mean(costos)),
                          "diversidad": float(np.std(poblacion))})
        if guardar_poblaciones:
            poblaciones.append(poblacion.copy())
        if g == generaciones:
            break
        # Elitismo: conservar los mejores individuos, sin modificarlos.
        indices = np.argsort(costos)[:elite]
        nueva = list(poblacion[indices].copy())
        while len(nueva) < n:
            p1 = elegir_padre(poblacion, costos, metodo, rng)
            p2 = elegir_padre(poblacion, costos, metodo, rng)
            # Cruce aritmético: x_hijo = alfa*p1 + (1-alfa)*p2.
            alfa = rng.uniform(0, 1)
            hijo = alfa*p1 + (1-alfa)*p2
            if rng.random() < pm:
                hijo += rng.normal(0, sigma)  # Mutación gaussiana
            nueva.append(float(np.clip(hijo, LIM_INF, LIM_SUP)))
        poblacion = np.asarray(nueva)
    return {"historial": pd.DataFrame(historial), "mejor_x": mejor_x,
            "mejor_f": mejor_global, "poblaciones": poblaciones,
            "evaluaciones": n*(generaciones+1)}

def comparar_experimentos(experimentos, repeticiones=8):
    resultados = {}
    filas = []
    # Repetimos con las mismas semillas en todos los experimentos.
    semillas = [NUMERO_LISTA*1000 + 31*i for i in range(repeticiones)]
    for nombre, cfg in experimentos.items():
        corridas = [algoritmo_genetico(cfg, s) for s in semillas]
        resultados[nombre] = corridas
        valores = np.array([r["mejor_f"] for r in corridas])
        filas.append({"Experimento": nombre, "Selección": cfg["seleccion"],
                      "Población": cfg["poblacion"], "Generaciones": cfg["generaciones"],
                      "P_mutación": cfg["probabilidad_mutacion"],
                      "σ_mutación": cfg["desviacion_mutacion"], "Elitismo": cfg["elitismo"],
                      "Error medio final": round(float(valores.mean()), 6),
                      "Error peor corrida": round(float(valores.max()), 6),
                      "Evaluaciones por corrida": corridas[0]["evaluaciones"]})
    return pd.DataFrame(filas), resultados

def graficar_comparacion(resultados, titulo="Comparación de convergencia"):
    fig, axes = plt.subplots(1, 2, figsize=(12,4.1))
    for nombre, corridas in resultados.items():
        # Todos los ensayos del mismo experimento tienen la misma longitud.
        curvas = np.stack([r["historial"]["mejor_acumulado"].to_numpy() for r in corridas])
        x = corridas[0]["historial"]["generacion"].to_numpy()
        mediana = np.median(curvas, axis=0)
        axes[0].plot(x, mediana, label=nombre)
        axes[1].plot(x, mediana, label=nombre)
    axes[0].set_title("Mediana del mejor error acumulado")
    axes[1].set_title("Detalle de valores pequeños (escala log)")
    axes[1].set_yscale("log")
    for ax in axes:
        ax.set_xlabel("Generación")
        ax.set_ylabel("f(x) = error (menor es mejor)")
        ax.grid(alpha=.3)
        ax.legend(fontsize=8)
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()

def graficar_poblacion(config, semilla):
    r = algoritmo_genetico(config, semilla, guardar_poblaciones=True)
    xs = np.linspace(LIM_INF, LIM_SUP, 500)
    mostrar = sorted(set([0, 1, 3, config["generaciones"] // 2, config["generaciones"]]))
    fig, axs = plt.subplots(1, len(mostrar), figsize=(3.2*len(mostrar),3), sharey=True)
    for ax,g in zip(np.atleast_1d(axs),mostrar):
        ax.plot(xs, objetivo(xs), lw=1.6)
        pob = r["poblaciones"][g]
        ax.scatter(pob, objetivo(pob), s=22, zorder=3)
        ax.set_title(f"Generación {g}")
        ax.set_xlabel("x")
        ax.grid(alpha=.25)
    np.atleast_1d(axs)[0].set_ylabel("f(x)")
    plt.tight_layout()
    plt.show()
    return r

print(f"Número de lista: {NUMERO_LISTA} | Función individual: f(x) = (x - {a:.2f})²")
print(f"Mínimo matemático conocido: x = {a:.2f}, f(x) = 0")


## Parámetros que puedes cambiar

| Parámetro | Opciones válidas | ¿Qué controla? |
|---|---|---|
| `seleccion` | `"torneo"` o `"ruleta"` | Cómo se eligen padres |
| `poblacion` | 4 a 100 | Número de soluciones por generación |
| `generaciones` | 1 a 300 | Cuántos ciclos se ejecutan |
| `probabilidad_mutacion` | 0 a 1 | Frecuencia de mutación por hijo |
| `desviacion_mutacion` | 0 a 5 | Magnitud de perturbación gaussiana |
| `elitismo` | 0 hasta población−1 | Individuos que se conservan sin cambios |

Se utiliza siempre **cruce aritmético** y **mutación gaussiana** para que las comparaciones tengan una base común. El método de selección sí puede cambiar.

**Consejo:** no aumentes todos los parámetros sin razón: poblaciones mayores y más generaciones requieren más evaluaciones. Trata de encontrar un equilibrio.


## 2. Diseña tres propuestas y compáralas

In [ ]:
# CONFIGURACIÓN DE REFERENCIA: NO MODIFICAR
BASE = {"seleccion":"torneo", "poblacion":8, "generaciones":12,
        "probabilidad_mutacion":0.30, "desviacion_mutacion":0.38,
        "elitismo":1}

# PROPUESTAS DEL ALUMNO: CAMBIA SOLAMENTE LOS VALORES ENTRE COMILLAS O LOS NÚMEROS.
# Debes modificar por lo menos DOS parámetros de BASE en CADA propuesta.
# Los valores de ejemplo sirven para mostrarte cómo se llenan; REEMPLÁZALOS.
P1 = {"seleccion":"ruleta", "poblacion":8, "generaciones":20,
      "probabilidad_mutacion":0.30, "desviacion_mutacion":0.38, "elitismo":1}

P2 = {"seleccion":"torneo", "poblacion":16, "generaciones":12,
      "probabilidad_mutacion":0.55, "desviacion_mutacion":0.38, "elitismo":1}

P3 = {"seleccion":"ruleta", "poblacion":12, "generaciones":25,
      "probabilidad_mutacion":0.40, "desviacion_mutacion":0.70, "elitismo":2}

claves = list(BASE.keys())
for nombre, p in {"P1":P1, "P2":P2, "P3":P3}.items():
    cambios = [k for k in claves if p[k] != BASE[k]]
    assert len(cambios) >= 2, f"{nombre}: debes modificar al menos dos parámetros"
assert len({tuple(p[k] for k in claves) for p in [P1,P2,P3]}) == 3, "Las propuestas deben ser distintas"

experimentos = {"BASE":BASE, "P1":P1, "P2":P2, "P3":P3}
resumen, resultados = comparar_experimentos(experimentos, repeticiones=8)
display(resumen)
graficar_comparacion(resultados, "Comparación de mis tres propuestas contra BASE")


## Reporte de interpretación

**No hay que escribir código**. Entrega el cuaderno con celdas ejecutadas y responde debajo.

**1. Diseño de propuestas.** Anota cuáles parámetros modificaste en P1, P2 y P3, comparados con BASE. Escribe una breve hipótesis: ¿por qué esperabas una mejora?

**2. Resultados y evidencia.** Transcribe de la tabla para BASE y tus tres propuestas: *error medio final*, *error peor corrida* y *evaluaciones por corrida*. No basta con decir «mejoró».

**3. Interpretación de convergencia.** Explica qué indican las curvas, cuál desciende más rápido y si alguna se estanca. Recuerda que la mediana de 8 ejecuciones no es una garantía del resultado de la siguiente ejecución.

**4. Decisión justificada.** Elige una configuración y justifica tu elección considerando tanto la calidad de la solución como el número de evaluaciones. Puedes elegir una que no tenga el menor error si ahorra mucho esfuerzo.

**5. Reflexión.** ¿Por qué no es suficiente ejecutar una sola vez el algoritmo para decidir cuáles son los mejores parámetros?

### Rúbrica
| Criterio | Puntos |
|---|---:|
| Tres propuestas distintas, con al menos dos cambios cada una e hipótesis | 1 |
| Tabla de resultados completa y correcta | 1 |
| Interpretación de curvas de convergencia | 1 |
| Selección y justificación numérica | 1 |
| Reporte de interpretación | 1 |
| **Total** | **5** |

**Observación:** son comparaciones experimentales, no una búsqueda garantizada de los «parámetros óptimos». Explica tus hallazgos en términos de **mejor configuración observada** bajo las pruebas realizadas.
